# 06 — GPU-Accelerated Validation + Notion Summary (v5)

**Run on H100 GPU node.**

Uses `validate_cross_pipeline_gpu_v5.py` which organizes comparisons into three groups:
- **MINIMAL**: within minimal-workflow comparisons
- **FULL**: within full-workflow comparisons
- **CROSS**: same platform, minimal vs full

For Mouse Brain 1M, creates a 1M-cell canonical subset to avoid OOM.

In [1]:
import cupy as cp
print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"VRAM: {cp.cuda.runtime.memGetInfo()[1] / 1e9:.1f} GB")

GPU: NVIDIA H100 NVL
VRAM: 99.9 GB


In [2]:
import json, os, time, glob
import pandas as pd
import numpy as np
import scanpy as sc

CONFIG_PATH = "benchmark_config.json"
DATASETS = ["pbmc3k", "lung65k", "mouse_brain_1m"]

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

Scanpy: 1.12


/tmp/ipykernel_1367554/4214173363.py:12: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


## Step 0: Prepare 1M-cell canonical subset for Mouse Brain

The full 1.15M x 22K matrix exceeds H100 VRAM when dense.
rsc and ScaleSC only processed 1M cells anyway, so validation only needs 1M.

In [3]:
mouse_cfg = cfg["datasets"]["mouse_brain_1m"]
full_h5ad = mouse_cfg["canonical_h5ad"]
subset_h5ad = full_h5ad.replace(".h5ad", "_1M.h5ad")

if not os.path.exists(subset_h5ad):
    print(f"Creating 1M subset from {full_h5ad}...")
    adata = sc.read_h5ad(full_h5ad)
    print(f"  Full: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    adata = adata[:1_000_000, :].copy()
    adata.write_h5ad(subset_h5ad)
    print(f"  Saved: {subset_h5ad} ({adata.n_obs:,} cells)")
    del adata
else:
    print(f"Already exists: {subset_h5ad}")

# Update config to point to 1M subset for validation
cfg["datasets"]["mouse_brain_1m"]["canonical_h5ad"] = subset_h5ad

# Write temp config
TEMP_CONFIG = "benchmark_config_validation.json"
with open(TEMP_CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)
print(f"Wrote temp config: {TEMP_CONFIG}")

Already exists: write/mouse_1m_canonical_filtered_1M.h5ad
Wrote temp config: benchmark_config_validation.json


## Step 1: Run v5 validation for all datasets

Runs all three groups (minimal, full, cross) for each dataset.
Pipelines not found on disk are automatically skipped.

In [4]:
import subprocess, sys

SCRIPT = "validate_cross_pipeline_gpu_v5.py"

validation_results = {}
for ds in DATASETS:
    print(f"\n{'='*72}")
    print(f"GPU Validation: {ds}")
    print(f"{'='*72}")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, SCRIPT,
         "--dataset", ds, "--config", TEMP_CONFIG, "--gpu"],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    validation_results[ds] = {"returncode": result.returncode, "elapsed": elapsed}

    if result.returncode == 0:
        print(result.stdout)
        print(f"Completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    else:
        print(f"FAILED (return code {result.returncode})")
        print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
        print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")

print(f"\n{'='*72}")
print("Validation timing summary:")
total_time = 0
for ds, r in validation_results.items():
    status = "OK" if r["returncode"] == 0 else "FAILED"
    print(f"  {ds:20s}: {r['elapsed']:8.1f}s ({r['elapsed']/60:.1f} min) [{status}]")
    total_time += r["elapsed"]
print(f"  {'TOTAL':20s}: {total_time:8.1f}s ({total_time/60:.1f} min)")


GPU Validation: pbmc3k
GPU mode: RMM initialized

Validation — PBMC3k

Minimal pipelines (5):
  scanpy_cpu                write/pbmc3k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu_0141              write/pbmc3k_rsc_gpu_0141_harmonized_clusters.csv
  rsc_gpu_015               write/pbmc3k_rsc_gpu_015_harmonized_clusters.csv
  seurat_cpu                write/pbmc3k_seurat_cpu_harmonized_clusters.csv
  scalesc_gpu               write/pbmc3k_scalesc_gpu_harmonized_clusters.csv

Full pipelines (0):

  Standardized DE: scanpy_cpu                (GPU) ... 2.2s
  Standardized DE: rsc_gpu_0141              (GPU) ... 1.4s
  Standardized DE: rsc_gpu_015               (GPU) ... 1.4s
  Standardized DE: seurat_cpu                (GPU) ... 1.5s
  Standardized DE: scalesc_gpu               (GPU) ... 1.4s

  Computing module scores ... 2.1s

  GROUP: MINIMAL (within-workflow)
    scanpy_cpu vs rsc_gpu_0141 ... 0.2s  ARI=0.923  Module ρ=1.0
    scanpy_cpu vs rsc_gpu_015 ... 0.2s  ARI=0.923  Module ρ=1.

## Step 2: Load all results

In [5]:
# Reload original config for display
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

all_summaries = []
all_details = {}

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]

    # v5 writes combined results across all groups
    combined_csv = f"write/validation_{prefix}_all_groups_summary.csv"
    combined_json = f"write/validation_{prefix}_all_groups_details.json"

    if os.path.exists(combined_csv):
        df = pd.read_csv(combined_csv)
        all_summaries.append(df)
        print(f"Loaded {ds}: {len(df)} comparisons")
    else:
        print(f"MISSING: {combined_csv}")

    if os.path.exists(combined_json):
        with open(combined_json) as f:
            all_details[ds] = json.load(f)

if all_summaries:
    full_summary = pd.concat(all_summaries, ignore_index=True)
    print(f"\nTotal comparisons across all datasets: {len(full_summary)}")

    # Show breakdown by group
    if "group" in full_summary.columns:
        print("\nBy group:")
        for grp, gdf in full_summary.groupby("group"):
            print(f"  {grp:10s}: {len(gdf)} comparisons")
else:
    full_summary = pd.DataFrame()

Loaded pbmc3k: 10 comparisons
Loaded lung65k: 10 comparisons
Loaded mouse_brain_1m: 5 comparisons

Total comparisons across all datasets: 25

By group:
  cross     : 1 comparisons
  full      : 3 comparisons
  minimal   : 21 comparisons


## Step 3: Per-dataset comparison tables (grouped by workflow)

In [6]:
for ds in DATASETS:
    if ds not in all_details:
        print(f"\nSkipping {ds} (no details)")
        continue

    ds_name = cfg["datasets"][ds]["name"]
    print(f"\n{'='*72}")
    print(f"{ds_name}")
    print(f"{'='*72}")

    # Get group info from summary
    ds_summary = full_summary[full_summary["dataset"] == ds_name] if not full_summary.empty else pd.DataFrame()
    group_lookup = {}
    if not ds_summary.empty and "group" in ds_summary.columns:
        for _, row in ds_summary.iterrows():
            parts = row["comparison"].split(" vs ")
            if len(parts) == 2:
                key = f"{parts[0]}__vs__{parts[1]}"
                group_lookup[key] = row["group"]

    rows = []
    for pair_key, metrics in all_details[ds].items():
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        group = group_lookup.get(pair_key, "?")
        rows.append({
            "Comparison": f"{parts[0]} vs {parts[1]}",
            "Group": group,
            "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Dice": round(metrics.get("mean_dice", 0), 3),
            "Std DEG Jac": round(metrics["mean_standardized_deg_jaccard"], 3) if metrics.get("mean_standardized_deg_jaccard") else "-",
            "Std DEG rho": round(metrics["mean_standardized_deg_spearman"], 3) if metrics.get("mean_standardized_deg_spearman") else "-",
            "Nat DEG rho": round(metrics["mean_native_deg_spearman"], 3) if metrics.get("mean_native_deg_spearman") else "-",
            "Module rho": round(metrics["mean_module_profile_rho"], 3) if metrics.get("mean_module_profile_rho") else "-",
        })

    if rows:
        df = pd.DataFrame(rows)
        order = {"minimal": 0, "full": 1, "cross": 2, "?": 3}
        df["_sort"] = df["Group"].map(order)
        df = df.sort_values("_sort").drop(columns="_sort")

        with pd.option_context("display.max_columns", None, "display.width", 200):
            print(df.to_string(index=False))


PBMC3k
                 Comparison   Group Clusters   ARI   NMI  Dice  Std DEG Jac  Std DEG rho Nat DEG rho  Module rho
 scanpy_cpu vs rsc_gpu_0141 minimal   9 vs 9 0.923 0.935 0.980        0.943        0.983       0.983       1.000
  scanpy_cpu vs rsc_gpu_015 minimal   9 vs 9 0.923 0.935 0.980        0.943        0.983       0.983       1.000
   scanpy_cpu vs seurat_cpu minimal  9 vs 10 0.655 0.762 0.790        0.736        0.830       0.877       0.980
  scanpy_cpu vs scalesc_gpu minimal   9 vs 9 0.761 0.824 0.835        0.778        0.851           -       0.984
rsc_gpu_0141 vs rsc_gpu_015 minimal   9 vs 9 1.000 1.000 1.000        1.000        1.000         1.0       1.000
 rsc_gpu_0141 vs seurat_cpu minimal  9 vs 10 0.639 0.750 0.782        0.725        0.821       0.874       0.980
rsc_gpu_0141 vs scalesc_gpu minimal   9 vs 9 0.748 0.816 0.829        0.773        0.849           -       0.984
  rsc_gpu_015 vs seurat_cpu minimal  9 vs 10 0.639 0.750 0.782        0.725        0.821

## Step 4: Runtime comparison

In [7]:
print("Runtime comparison")
print("=" * 72)

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    ds_name = cfg["datasets"][ds]["name"]
    spec_files = sorted(glob.glob(f"write/{prefix}_*_spec.json"))

    if not spec_files:
        continue

    print(f"\n--- {ds_name} ---")
    for sf in spec_files:
        with open(sf) as f:
            spec = json.load(f)
        pipeline = spec.get("pipeline", os.path.basename(sf))
        workflow = spec.get("workflow", "minimal" if "harmonized" in sf else "full" if "full" in sf else "?")
        timings = spec.get("timings_seconds", {})
        total = sum(timings.values()) if timings else 0
        n_cells = spec.get("results", {}).get("n_cells", "?")
        n_clusters = spec.get("results", {}).get("n_clusters", "?")
        if total > 3600:
            t_str = f"{total:8.1f}s ({total/3600:.1f} hr)"
        else:
            t_str = f"{total:8.1f}s ({total/60:.1f} min)"
        print(f"  {pipeline:35s} [{workflow:7s}] | {str(n_cells):>10s} cells | {str(n_clusters):>3s} cl | {t_str}")

Runtime comparison

--- PBMC3k ---
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  scalesc_gpu_harmonized              [minimal] |       2638 cells |   9 cl |      3.4s (0.1 min)
  scanpy_cpu_harmonized               [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  seurat_cpu_harmonized               [minimal] |       2638 cells |  10 cl |      0.0s (0.0 min)

--- Human Lung Cell 65K ---
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  37 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  38 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  37 cl |      0.0s (0.0 min)
  scalesc_gpu_harmonized              [minimal] |     

## Step 5: Mouse Brain GPU speedup (step-by-step)

In [8]:
prefix = cfg["datasets"]["mouse_brain_1m"]["pipeline_prefix"]

# Try full workflow specs first, then minimal
spec_pairs = [
    ("Full workflow", f"write/{prefix}_scanpy_cpu_full_spec.json", f"write/{prefix}_rsc_gpu_0141_full_spec.json"),
    ("Minimal workflow", f"write/{prefix}_scanpy_cpu_harmonized_spec.json", f"write/{prefix}_rsc_gpu_0141_harmonized_spec.json"),
]

for label, scanpy_path, rsc_path in spec_pairs:
    if not (os.path.exists(scanpy_path) and os.path.exists(rsc_path)):
        continue

    with open(scanpy_path) as f:
        s_spec = json.load(f)
    with open(rsc_path) as f:
        r_spec = json.load(f)

    s_t = s_spec.get("timings_seconds", {})
    r_t = r_spec.get("timings_seconds", {})

    print(f"Mouse Brain: Scanpy CPU vs rsc GPU ({label})")
    print(f"  Scanpy cells: {s_spec.get('results',{}).get('n_cells','?'):,}")
    print(f"  rsc cells:    {rSkip to main panel
import subprocess, sys

SCRIPT = "validate_cross_pipeline_gpu_v5.py"

validation_results = {}
for ds in DATASETS:
    print(f"\n{'='*72}")
    print(f"GPU Validation: {ds}")
    print(f"{'='*72}")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, SCRIPT,
         "--dataset", ds, "--config", TEMP_CONFIG, "--gpu"],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    validation_results[ds] = {"returncode": result.returncode, "elapsed": elapsed}

    if result.returncode == 0:
        print(result.stdout)
        print(f"Completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    else:
        print(f"FAILED (return code {result.returncode})")
        print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
        print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")

print(f"\n{'='*72}")
print("Validation timing summary:")
total_time = 0
for ds, r in validation_results.items():
    status = "OK" if r["returncode"] == 0 else "FAILED"
    print(f"  {ds:20s}: {r['elapsed']:8.1f}s ({r['elapsed']/60:.1f} min) [{status}]")
    total_time += r["elapsed"]
print(f"  {'TOTAL':20s}: {total_time:8.1f}s ({total_time/60:.1f} min)")


========================================================================
GPU Validation: pbmc3k
========================================================================
GPU mode: RMM initialized

========================================================================
Validation — PBMC3k
========================================================================

Minimal pipelines (5):
  scanpy_cpu                write/pbmc3k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu_0141              write/pbmc3k_rsc_gpu_0141_harmonized_clusters.csv
  rsc_gpu_015               write/pbmc3k_rsc_gpu_015_harmonized_clusters.csv
  seurat_cpu                write/pbmc3k_seurat_cpu_harmonized_clusters.csv
  scalesc_gpu               write/pbmc3k_scalesc_gpu_harmonized_clusters.csv

Full pipelines (0):

  Standardized DE: scanpy_cpu                (GPU) ... 2.2s
  Standardized DE: rsc_gpu_0141              (GPU) ... 1.4s
  Standardized DE: rsc_gpu_015               (GPU) ... 1.4s
  Standardized DE: seurat_cpu                (GPU) ... 1.5s
  Standardized DE: scalesc_gpu               (GPU) ... 1.4s

  Computing module scores ... 2.1s

========================================================================
  GROUP: MINIMAL (within-workflow)
========================================================================
    scanpy_cpu vs rsc_gpu_0141 ... 0.2s  ARI=0.923  Module ρ=1.0
    scanpy_cpu vs rsc_gpu_015 ... 0.2s  ARI=0.923  Module ρ=1.0
    scanpy_cpu vs seurat_cpu ... 0.2s  ARI=0.655  Module ρ=0.9801587301587303
    scanpy_cpu vs scalesc_gpu ... 0.1s  ARI=0.761  Module ρ=0.9841269841269842
    rsc_gpu_0141 vs rsc_gpu_015 ... 0.2s  ARI=1.000  Module ρ=1.0
    rsc_gpu_0141 vs seurat_cpu ... 0.2s  ARI=0.639  Module ρ=0.9801587301587301
    rsc_gpu_0141 vs scalesc_gpu ... 0.1s  ARI=0.748  Module ρ=0.9841269841269842
    rsc_gpu_015 vs seurat_cpu ... 0.1s  ARI=0.639  Module ρ=0.9801587301587301
    rsc_gpu_015 vs scalesc_gpu ... 0.1s  ARI=0.748  Module ρ=0.9841269841269842
    seurat_cpu vs scalesc_gpu ... 0.1s  ARI=0.662  Module ρ=0.9920634920634921

────────────────────────────────────────────────────────────────────────
  PBMC3k — MINIMAL
────────────────────────────────────────────────────────────────────────
                 comparison  clusters_a  clusters_b   ARI   NMI  mean_dice  mean_module_profile_rho  mean_standardized_deg_jaccard  mean_standardized_deg_spearman
 scanpy_cpu vs rsc_gpu_0141           9           9 0.923 0.935      0.980                    1.000                          0.943                           0.983
  scanpy_cpu vs rsc_gpu_015           9           9 0.923 0.935      0.980                    1.000                          0.943                           0.983
   scanpy_cpu vs seurat_cpu           9          10 0.655 0.762      0.790                    0.980                          0.736                           0.830
  scanpy_cpu vs scalesc_gpu           9           9 0.761 0.824      0.835                    0.984                          0.778                           0.851
rsc_gpu_0141 vs rsc_gpu_015           9           9 1.000 1.000      1.000                    1.000                          1.000                           1.000
 rsc_gpu_0141 vs seurat_cpu           9          10 0.639 0.750      0.782                    0.980                          0.725                           0.821
rsc_gpu_0141 vs scalesc_gpu           9           9 0.748 0.816      0.829                    0.984                          0.773                           0.849
  rsc_gpu_015 vs seurat_cpu           9          10 0.639 0.750      0.782                    0.980                          0.725                           0.821
 rsc_gpu_015 vs scalesc_gpu           9           9 0.748 0.816      0.829                    0.984                          0.773                           0.849
  seurat_cpu vs scalesc_gpu          10           9 0.662 0.763      0.857                    0.992                          0.830                           0.848

  Skipping FULL group: fewer than 2 pipelines detected.

  Skipping CROSS group: no matching minimal/full platform pairs.

========================================================================
  OUTPUT FILES
========================================================================
  Combined summary : write/validation_pbmc3k_all_groups_summary.csv
  Combined details : write/validation_pbmc3k_all_groups_details.json
  Notion markdown  : write/validation_pbmc3k_notion.md
  minimal  dir     : write/validation_pbmc3k_minimal/
  Standardized DE  : write/validation_pbmc3k_standardized_de/

Completed in 15.0s (0.3 min)

========================================================================
GPU Validation: lung65k
========================================================================
GPU mode: RMM initialized

========================================================================
Validation — Human Lung Cell 65K
========================================================================

Minimal pipelines (5):
  scanpy_cpu                write/lung_65k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu_0141              write/lung_65k_rsc_gpu_0141_harmonized_clusters.csv
  rsc_gpu_015               write/lung_65k_rsc_gpu_015_harmonized_clusters.csv
  seurat_cpu                write/lung_65k_seurat_cpu_harmonized_clusters.csv
  scalesc_gpu               write/lung_65k_scalesc_gpu_harmonized_clusters.csv

Full pipelines (0):

  Standardized DE: scanpy_cpu                (GPU) ... 16.4s
  Standardized DE: rsc_gpu_0141              (GPU) ... 14.9s
  Standardized DE: rsc_gpu_015               (GPU) ... 15.1s
  Standardized DE: seurat_cpu                (GPU) ... 16.1s
  Standardized DE: scalesc_gpu               (GPU) ... 15.1s

  Computing module scores ... 12.2s

========================================================================
  GROUP: MINIMAL (within-workflow)
========================================================================
    scanpy_cpu vs rsc_gpu_0141 ... 1.7s  ARI=0.873  Module ρ=0.9675675675675675
    scanpy_cpu vs rsc_gpu_015 ... 1.7s  ARI=0.902  Module ρ=1.0
    scanpy_cpu vs seurat_cpu ... 1.6s  ARI=0.704  Module ρ=0.9157894736842105
    scanpy_cpu vs scalesc_gpu ... 1.2s  ARI=0.786  Module ρ=0.9578947368421052
    rsc_gpu_0141 vs rsc_gpu_015 ... 1.6s  ARI=0.927  Module ρ=0.972972972972973
    rsc_gpu_0141 vs seurat_cpu ... 1.5s  ARI=0.726  Module ρ=0.9513513513513514
    rsc_gpu_0141 vs scalesc_gpu ... 1.2s  ARI=0.773  Module ρ=0.9567567567567568
    rsc_gpu_015 vs seurat_cpu ... 1.6s  ARI=0.718  Module ρ=0.9157894736842105
    rsc_gpu_015 vs scalesc_gpu ... 1.2s  ARI=0.756  Module ρ=0.9578947368421055
    seurat_cpu vs scalesc_gpu ... 1.2s  ARI=0.720  Module ρ=0.9315789473684211

────────────────────────────────────────────────────────────────────────
  Human Lung Cell 65K — MINIMAL
────────────────────────────────────────────────────────────────────────
                 comparison  clusters_a  clusters_b   ARI   NMI  mean_dice  mean_module_profile_rho  mean_standardized_deg_jaccard  mean_standardized_deg_spearman
 scanpy_cpu vs rsc_gpu_0141          38          37 0.873 0.931      0.844                    0.968                          0.835                           0.914
  scanpy_cpu vs rsc_gpu_015          38          38 0.902 0.942      0.909                    1.000                          0.890                           0.966
   scanpy_cpu vs seurat_cpu          38          44 0.704 0.865      0.697                    0.916                          0.700                           0.843
  scanpy_cpu vs scalesc_gpu          38          38 0.786 0.904      0.778                    0.958                          0.783                           0.889
rsc_gpu_0141 vs rsc_gpu_015          37          38 0.927 0.963      0.882                    0.973                          0.880                           0.937
 rsc_gpu_0141 vs seurat_cpu          37          44 0.726 0.870      0.784                    0.951                          0.781                           0.893
rsc_gpu_0141 vs scalesc_gpu          37          38 0.773 0.894      0.754                    0.957                          0.744                           0.865
  rsc_gpu_015 vs seurat_cpu          38          44 0.718 0.869      0.730                    0.916                          0.729                           0.857
 rsc_gpu_015 vs scalesc_gpu          38          38 0.756 0.892      0.787                    0.958                          0.776                           0.897
  seurat_cpu vs scalesc_gpu          44          38 0.720 0.854      0.711                    0.932                          0.723                           0.865

  Skipping FULL group: fewer than 2 pipelines detected.

  Skipping CROSS group: no matching minimal/full platform pairs.

========================================================================
  OUTPUT FILES
========================================================================
  Combined summary : write/validation_lung_65k_all_groups_summary.csv
  Combined details : write/validation_lung_65k_all_groups_details.json
  Notion markdown  : write/validation_lung_65k_notion.md
  minimal  dir     : write/validation_lung_65k_minimal/
  Standardized DE  : write/validation_lung_65k_standardized_de/

Completed in 110.4s (1.8 min)

========================================================================
GPU Validation: mouse_brain_1m
========================================================================
GPU mode: RMM initialized

========================================================================
Validation — Mouse Brain 1.3M
========================================================================

Minimal pipelines (2):
  scanpy_cpu                write/mouse_1m_scanpy_cpu_harmonized_clusters.csv
  scalesc_gpu               write/mouse_1m_scalesc_gpu_harmonized_clusters.csv

Full pipelines (3):
  scanpy_cpu_full           write/mouse_1m_scanpy_cpu_full_clusters.csv
  rsc_gpu_0141_full         write/mouse_1m_rsc_gpu_0141_full_clusters.csv
  rsc_gpu_015_full          write/mouse_1m_rsc_gpu_015_full_clusters.csv

  Standardized DE: scanpy_cpu                (GPU) ... 80.1s
  Standardized DE: scalesc_gpu               (GPU) ... 73.7s
  Standardized DE: scanpy_cpu_full           (GPU) ... 75.6s
  Standardized DE: rsc_gpu_0141_full         (GPU) ... 77.0s
  Standardized DE: rsc_gpu_015_full          (GPU) ... 77.1s

  Computing module scores ... 139.2s

========================================================================
  GROUP: MINIMAL (within-workflow)
========================================================================
    scanpy_cpu vs scalesc_gpu ... 15.8s  ARI=0.497  Module ρ=0.978835978835979

────────────────────────────────────────────────────────────────────────
  Mouse Brain 1.3M — MINIMAL
────────────────────────────────────────────────────────────────────────
               comparison  clusters_a  clusters_b   ARI   NMI  mean_dice  mean_module_profile_rho  mean_standardized_deg_jaccard  mean_standardized_deg_spearman
scanpy_cpu vs scalesc_gpu          30          27 0.497 0.747      0.798                    0.979                          0.831                           0.934

========================================================================
  GROUP: FULL (within-workflow)
========================================================================
    scanpy_cpu_full vs rsc_gpu_0141_full ... 16.8s  ARI=0.529  Module ρ=0.9928571428571429
    scanpy_cpu_full vs rsc_gpu_015_full ... 16.5s  ARI=0.529  Module ρ=0.9928571428571429
    rsc_gpu_0141_full vs rsc_gpu_015_full ... 17.6s  ARI=1.000  Module ρ=1.0

────────────────────────────────────────────────────────────────────────
  Mouse Brain 1.3M — FULL
────────────────────────────────────────────────────────────────────────
                           comparison  clusters_a  clusters_b   ARI   NMI  mean_dice  mean_module_profile_rho  mean_standardized_deg_jaccard  mean_standardized_deg_spearman
 scanpy_cpu_full vs rsc_gpu_0141_full          32          36 0.529 0.759      0.752                    0.993                          0.805                           0.913
  scanpy_cpu_full vs rsc_gpu_015_full          32          36 0.529 0.759      0.752                    0.993                          0.805                           0.913
rsc_gpu_0141_full vs rsc_gpu_015_full          36          36 1.000 1.000      1.000                    1.000                          1.000                           1.000

========================================================================
  GROUP: CROSS (minimal vs full, same platform)
========================================================================
    scanpy_cpu vs scanpy_cpu_full ... 16.5s  ARI=0.530  Module ρ=0.9161904761904762

────────────────────────────────────────────────────────────────────────
  Mouse Brain 1.3M — CROSS
────────────────────────────────────────────────────────────────────────
                   comparison  clusters_a  clusters_b   ARI   NMI  mean_dice  mean_module_profile_rho  mean_standardized_deg_jaccard  mean_standardized_deg_spearman
scanpy_cpu vs scanpy_cpu_full          30          32 0.530 0.733      0.730                    0.916                          0.789                           0.873

========================================================================
  OUTPUT FILES
========================================================================
  Combined summary : write/validation_mouse_1m_all_groups_summary.csv
  Combined details : write/validation_mouse_1m_all_groups_details.json
  Notion markdown  : write/validation_mouse_1m_notion.md
  minimal  dir     : write/validation_mouse_1m_minimal/
  full     dir     : write/validation_mouse_1m_full/
  cross    dir     : write/validation_mouse_1m_cross/
  Standardized DE  : write/validation_mouse_1m_standardized_de/

Completed in 620.9s (10.3 min)

========================================================================
Validation timing summary:
  pbmc3k              :     15.0s (0.3 min) [OK]
  lung65k             :    110.4s (1.8 min) [OK]
  mouse_brain_1m      :    620.9s (10.3 min) [OK]
  TOTAL               :    746.3s (12.4 min)

print("Runtime comparison")
print("=" * 72)

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    ds_name = cfg["datasets"][ds]["name"]
    spec_files = sorted(glob.glob(f"write/{prefix}_*_spec.json"))

    if not spec_files:
        continue

    print(f"\n--- {ds_name} ---")
    for sf in spec_files:
        with open(sf) as f:
            spec = json.load(f)
        pipeline = spec.get("pipeline", os.path.basename(sf))
        workflow = spec.get("workflow", "minimal" if "harmonized" in sf else "full" if "full" in sf else "?")
        timings = spec.get("timings_seconds", {})
        total = sum(timings.values()) if timings else 0
        n_cells = spec.get("results", {}).get("n_cells", "?")
        n_clusters = spec.get("results", {}).get("n_clusters", "?")
        if total > 3600:
            t_str = f"{total:8.1f}s ({total/3600:.1f} hr)"
        else:
            t_str = f"{total:8.1f}s ({total/60:.1f} min)"
        print(f"  {pipeline:35s} [{workflow:7s}] | {str(n_cells):>10s} cells | {str(n_clusters):>3s} cl | {t_str}")

Runtime comparison
========================================================================

--- PBMC3k ---
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  scalesc_gpu_harmonized              [minimal] |       2638 cells |   9 cl |      3.4s (0.1 min)
  scanpy_cpu_harmonized               [minimal] |       2638 cells |   9 cl |      0.0s (0.0 min)
  seurat_cpu_harmonized               [minimal] |       2638 cells |  10 cl |      0.0s (0.0 min)

--- Human Lung Cell 65K ---
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  37 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  38 cl |      0.0s (0.0 min)
  rsc_gpu_harmonized                  [minimal] |      65462 cells |  37 cl |      0.0s (0.0 min)
  scalesc_gpu_harmonized              [minimal] |      65462 cells |  38 cl |      5.4s (0.1 min)
  scanpy_cpu_harmonized               [minimal] |      65462 cells |  38 cl |      0.0s (0.0 min)
  seurat_cpu_harmonized               [minimal] |      65462 cells |  44 cl |      0.0s (0.0 min)

--- Mouse Brain 1.3M ---
  rsc_gpu_0141_full                   [FULL (with regress_out + scale)] |    1000000 cells |  36 cl |    614.8s (10.2 min)
  rsc_gpu_015_full                    [FULL (with regress_out + scale)] |    1000000 cells |  36 cl |   1402.0s (23.4 min)
  rsc_gpu_harmonized                  [minimal] |    1000000 cells |  30 cl |    491.8s (8.2 min)
  scalesc_gpu_harmonized              [minimal] |    1000000 cells |  27 cl |     51.9s (0.9 min)
  scanpy_cpu_full                     [FULL (with regress_out + scale)] |    1000000 cells |  32 cl |   3574.6s (59.6 min)
  scanpy_cpu_harmonized               [minimal] |    1153539 cells |  30 cl |  19974.3s (5.5 hr)

Step 7: Save CSVs + cleanup
# Save consolidated summary
if all_summaries:
    full_summary.to_csv("write/all_datasets_validation_summary.csv", index=False)
    print("Saved: write/all_datasets_validation_summary.csv")

# Clean up temp config
if os.path.exists(TEMP_CONFIG):
    os.remove(TEMP_CONFIG)
    print(f"Cleaned up: {TEMP_CONFIG}")

print("\nAll validation outputs:")
for d in sorted(glob.glob("write/validation_*/")):
    n_files = len(glob.glob(d + "*"))
    print(f"  {d:50s} ({n_files} files)")

# v5 auto-generated notion markdown
notion_files = sorted(glob.glob("write/validation_*_notion.md"))
if notion_files:
    print("\nNotion markdown files (auto-generated by v5):")
    for nf in notion_files:
        print(f"  {nf}")

print("\nDone!")

Saved: write/all_datasets_validation_summary.csv
Cleaned up: benchmark_config_validation.json

All validation outputs:
  write/validation_lung_65k_harmonized/              (24 files)
  write/validation_lung_65k_minimal/                 (12 files)
  write/validation_lung_65k_standardized_de/         (5 files)
  write/validation_mouse_1m_cross/                   (3 files)
  write/validation_mouse_1m_full/                    (5 files)
  write/validation_mouse_1m_harmonized/              (8 files)
  write/validation_mouse_1m_minimal/                 (3 files)
  write/validation_mouse_1m_standardized_de/         (5 files)
  write/validation_pbmc3k_harmonized/                (24 files)
  write/validation_pbmc3k_minimal/                   (12 files)
  write/validation_pbmc3k_standardized_de/           (5 files)

Notion markdown files (auto-generated by v5):
  write/validation_lung_65k_notion.md
  write/validation_mouse_1m_notion.md
  write/validation_pbmc3k_notion.md

Done!
_spec.get('results',{}).get('n_cells','?'):,}")
    print()

    steps = ["normalize_log1p", "hvg", "regress_out", "scale", "pca", "neighbors", "leiden", "umap", "de"]
    rows = []
    for step in steps:
        s_time = s_t.get(step)
        r_time = r_t.get(step)
        if s_time and r_time and r_time > 0:
            rows.append({
                "Step": step,
                "Scanpy CPU (s)": round(s_time, 1),
                "rsc GPU (s)": round(r_time, 1),
                "Speedup": f"{s_time/r_time:.0f}x",
            })

    s_total = sum(s_t.get(s, 0) for s in steps)
    r_total = sum(r_t.get(s, 0) for s in steps)
    if r_total > 0:
        rows.append({
            "Step": "TOTAL (excl I/O)",
            "Scanpy CPU (s)": round(s_total, 1),
            "rsc GPU (s)": round(r_total, 1),
            "Speedup": f"{s_total/r_total:.0f}x",
        })

    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
    print()

# rsc version comparison
for ver_a, ver_b in [("0141", "015")]:
    pa = f"write/{prefix}_rsc_gpu_{ver_a}_full_spec.json"
    pb = f"write/{prefix}_rsc_gpu_{ver_b}_full_spec.json"
    if os.path.exists(pa) and os.path.exists(pb):
        with open(pa) as f:
            sa = json.load(f)
        with open(pb) as f:
            sb = json.load(f)
        ta = sa.get("timings_seconds", {})
        tb = sb.get("timings_seconds", {})
        print(f"rsc version comparison (full workflow):")
        print(f"  0.14.1 total: {sum(ta.values()):.1f}s  |  0.15.0rc5 total: {sum(tb.values()):.1f}s")
        print(f"  0.14.1 markers: {sa.get('results',{}).get('n_marker_rows_filtered','?')}  |  0.15.0rc5 markers: {sb.get('results',{}).get('n_marker_rows_filtered','?')}")

Mouse Brain: Scanpy CPU vs rsc GPU (Full workflow)
  Scanpy cells: 1,000,000
  rsc cells:    1,000,000

            Step  Scanpy CPU (s)  rsc GPU (s) Speedup
 normalize_log1p             8.0          2.2      4x
             hvg            41.5          7.1      6x
     regress_out            11.3          1.9      6x
           scale            25.4          1.9     13x
             pca            76.0         11.9      6x
       neighbors           338.5         12.5     27x
          leiden           872.8          3.0    290x
            umap           783.9          2.0    392x
              de           835.2          4.3    194x
TOTAL (excl I/O)          2992.6         46.9     64x

rsc version comparison (full workflow):
  0.14.1 total: 614.8s  |  0.15.0rc5 total: 1402.0s
  0.14.1 markers: 61557  |  0.15.0rc5 markers: 93295


## Step 6: Notion-ready markdown (copy-paste)

In [9]:
def df_to_md(df, title=""):
    lines = []
    if title:
        lines.append(f"### {title}\n")
    cols = df.columns.tolist()
    lines.append("| " + " | ".join(str(c) for c in cols) + " |")
    lines.append("| " + " | ".join("---" for _ in cols) + " |")
    for _, row in df.iterrows():
        vals = []
        for v in row.values:
            if isinstance(v, float):
                vals.append(f"{v:.3f}")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


print("\n" + "=" * 72)
print("COPY-PASTE FOR NOTION")
print("=" * 72)

for ds in DATASETS:
    if ds not in all_details:
        continue
    ds_name = cfg["datasets"][ds]["name"]

    # Get group info
    ds_summary = full_summary[full_summary["dataset"] == ds_name] if not full_summary.empty else pd.DataFrame()
    group_lookup = {}
    if not ds_summary.empty and "group" in ds_summary.columns:
        for _, row in ds_summary.iterrows():
            parts = row["comparison"].split(" vs ")
            if len(parts) == 2:
                group_lookup[f"{parts[0]}__vs__{parts[1]}"] = row["group"]

    for group_name in ["minimal", "full", "cross"]:
        rows = []
        for pair_key, metrics in all_details[ds].items():
            if group_lookup.get(pair_key, "?") != group_name:
                continue
            parts = pair_key.split("__vs__")
            if len(parts) != 2:
                continue
            rows.append({
                "Comparison": f"{parts[0]} vs {parts[1]}",
                "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
                "ARI": round(metrics["ARI"], 3),
                "NMI": round(metrics["NMI"], 3),
                "Dice": round(metrics.get("mean_dice", 0), 3),
                "DEG Jac": round(metrics.get("mean_standardized_deg_jaccard", 0), 3) if metrics.get("mean_standardized_deg_jaccard") else "-",
                "DEG rho": round(metrics.get("mean_standardized_deg_spearman", 0), 3) if metrics.get("mean_standardized_deg_spearman") else "-",
                "Module rho": round(metrics.get("mean_module_profile_rho", 0), 3) if metrics.get("mean_module_profile_rho") else "-",
            })

        if rows:
            print(f"\n{df_to_md(pd.DataFrame(rows), title=f'{ds_name} -- {group_name.upper()}')}")
            print()


COPY-PASTE FOR NOTION

### PBMC3k -- MINIMAL

| Comparison | Clusters | ARI | NMI | Dice | DEG Jac | DEG rho | Module rho |
| --- | --- | --- | --- | --- | --- | --- | --- |
| scanpy_cpu vs rsc_gpu_0141 | 9 vs 9 | 0.923 | 0.935 | 0.980 | 0.943 | 0.983 | 1.000 |
| scanpy_cpu vs rsc_gpu_015 | 9 vs 9 | 0.923 | 0.935 | 0.980 | 0.943 | 0.983 | 1.000 |
| scanpy_cpu vs seurat_cpu | 9 vs 10 | 0.655 | 0.762 | 0.790 | 0.736 | 0.830 | 0.980 |
| scanpy_cpu vs scalesc_gpu | 9 vs 9 | 0.761 | 0.824 | 0.835 | 0.778 | 0.851 | 0.984 |
| rsc_gpu_0141 vs rsc_gpu_015 | 9 vs 9 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 |
| rsc_gpu_0141 vs seurat_cpu | 9 vs 10 | 0.639 | 0.750 | 0.782 | 0.725 | 0.821 | 0.980 |
| rsc_gpu_0141 vs scalesc_gpu | 9 vs 9 | 0.748 | 0.816 | 0.829 | 0.773 | 0.849 | 0.984 |
| rsc_gpu_015 vs seurat_cpu | 9 vs 10 | 0.639 | 0.750 | 0.782 | 0.725 | 0.821 | 0.980 |
| rsc_gpu_015 vs scalesc_gpu | 9 vs 9 | 0.748 | 0.816 | 0.829 | 0.773 | 0.849 | 0.984 |
| seurat_cpu vs scalesc_gpu | 10 

## Step 7: Save CSVs + cleanup

In [10]:
# Save consolidated summary
if all_summaries:
    full_summary.to_csv("write/all_datasets_validation_summary.csv", index=False)
    print("Saved: write/all_datasets_validation_summary.csv")

# Clean up temp config
if os.path.exists(TEMP_CONFIG):
    os.remove(TEMP_CONFIG)
    print(f"Cleaned up: {TEMP_CONFIG}")

print("\nAll validation outputs:")
for d in sorted(glob.glob("write/validation_*/")):
    n_files = len(glob.glob(d + "*"))
    print(f"  {d:50s} ({n_files} files)")

# v5 auto-generated notion markdown
notion_files = sorted(glob.glob("write/validation_*_notion.md"))
if notion_files:
    print("\nNotion markdown files (auto-generated by v5):")
    for nf in notion_files:
        print(f"  {nf}")

print("\nDone!")

Saved: write/all_datasets_validation_summary.csv
Cleaned up: benchmark_config_validation.json

All validation outputs:
  write/validation_lung_65k_harmonized/              (24 files)
  write/validation_lung_65k_minimal/                 (12 files)
  write/validation_lung_65k_standardized_de/         (5 files)
  write/validation_mouse_1m_cross/                   (3 files)
  write/validation_mouse_1m_full/                    (5 files)
  write/validation_mouse_1m_harmonized/              (8 files)
  write/validation_mouse_1m_minimal/                 (3 files)
  write/validation_mouse_1m_standardized_de/         (5 files)
  write/validation_pbmc3k_harmonized/                (24 files)
  write/validation_pbmc3k_minimal/                   (12 files)
  write/validation_pbmc3k_standardized_de/           (5 files)

Notion markdown files (auto-generated by v5):
  write/validation_lung_65k_notion.md
  write/validation_mouse_1m_notion.md
  write/validation_pbmc3k_notion.md

Done!
